# Find all combinations of units

Across all scenarios, which patients have each combination of nearest unit and transfer unit in the base and the Taunton+Worcester MT case?

## Code setup

In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
dir_output = 'output'
# Scenario results:
file_results_comparison = 'results_full_comparison_taunworc_base'
# Output:
file_unit_combos = 'unit_combos.csv'

In [3]:
transfer_rates = [1.0, 0.67]  # should match existing results files.

## Load data

### Scenario comparison results

Load in these results files:

In [4]:
results_paths = {}
for s_bias in ['', '_bias10min']:
    f = f'{file_results_comparison}{s_bias}.csv'
    p = os.path.join(dir_output, f)
    label = 'standardbias' if len(s_bias) < 1 else s_bias.replace('_', '')
    results_paths[label] = p

In [5]:
results_paths

{'standardbias': 'output/results_full_comparison_taunworc_base.csv',
 'bias10min': 'output/results_full_comparison_taunworc_base_bias10min.csv'}

Only load in these columns with the chosen unit names:

In [6]:
cols_unit_postcodes = ['nearest_ivt_unit', 'transfer_unit']
cols_unit_postcodes = [f'{c}_{s}' for c in cols_unit_postcodes
                       for s in ['base', 'taunworc']]

For each file, gather the unique sets of units:

In [7]:
list_of_dfs = []
for k, p in results_paths.items():
    df = pd.read_csv(p, usecols=cols_unit_postcodes)
    df_unit_combos = df[cols_unit_postcodes].drop_duplicates(ignore_index=True)
    df_unit_combos[k] = 1
    list_of_dfs.append(df_unit_combos.copy())

Turn unit data into a DataFrame and remove duplicate entries:

In [8]:
df_unit_combos = pd.concat(list_of_dfs, axis='rows', ignore_index=True)
df_unit_combos = df_unit_combos.drop_duplicates(subset=cols_unit_postcodes,
                                                ignore_index=True)
# Convert tracking columns to integer:
df_unit_combos = df_unit_combos.fillna(0).convert_dtypes()

Label unit combinations.

Use labels that start with '42' so they're not confused with a standard range index.

In [9]:
df_unit_combos['unit_combo'] = np.arange(len(df_unit_combos)) + 42000
df_unit_combos = df_unit_combos.set_index('unit_combo')

Check results:

In [10]:
display(df_unit_combos.head(3))

,nearest_ivt_unit_base,nearest_ivt_unit_taunworc,transfer_unit_base,transfer_unit_taunworc,standardbias,bias10min
unit_combo,,,,,,
42000,BN25BE,BN25BE,BN25BE,BN25BE,1,0
42001,BN112DH,BN112DH,BN25BE,BN25BE,1,0
42002,CA27HY,CA27HY,NE14LP,NE14LP,1,0


In [11]:
print(len(df_unit_combos))

112


Save results:

In [12]:
p = os.path.join(dir_output, file_unit_combos)
df_unit_combos.to_csv(p, index=True)

## Check for changes

Check labels for units that change with scenario:

In [13]:
m1 = (df_unit_combos['nearest_ivt_unit_base'] !=
      df_unit_combos['nearest_ivt_unit_taunworc'])
m2 = (df_unit_combos['transfer_unit_base'] !=
      df_unit_combos['transfer_unit_taunworc'])

display(df_unit_combos.loc[m1 | m2])

,nearest_ivt_unit_base,nearest_ivt_unit_taunworc,transfer_unit_base,transfer_unit_taunworc,standardbias,bias10min
unit_combo,,,,,,
42057,WR51DD,WR51DD,B152TH,WR51DD,1,0
42060,GL537AN,GL537AN,BS105NB,WR51DD,1,0
42070,EX314JB,EX314JB,PL68DH,TA15DA,1,0
42083,TA15DA,TA15DA,BS105NB,TA15DA,1,0
42084,EX25DW,EX25DW,PL68DH,TA15DA,1,0
42086,DT12JY,DT12JY,SO166YD,TA15DA,1,0
42091,HR12ER,HR12ER,B152TH,WR51DD,1,0
42099,DT12JY,TA15DA,SO166YD,TA15DA,1,0
42100,DY12HQ,WR51DD,B152TH,WR51DD,1,0


For example, in the standard case (travel bias for MT unit is 0.1minutes, `standardbias` column is 1), the combination of nearest-transfer unit 'TA15DA'-'TA15DA' appears twice in the `taunworc` columns, but these patients must be kept separate for the comparison with the base scenario because of the difference in their `base` columns.